# Phase 3: Feature Selection (TRAINING DATA ONLY)
**RULE: Everything in this notebook runs on TRAINING data only.**

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(REPO_DIR, 'data')):
    REPO_DIR = os.getcwd()
sys.path.insert(0, REPO_DIR)

TRAIN = {l1: os.path.join(REPO_DIR, f'data/train/{l1}/kvl_shared_task_{l1}_train.csv') for l1 in ['es','de','cn']}
DEV = {l1: os.path.join(REPO_DIR, f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv') for l1 in ['es','de','cn']}
from features.pipeline import FeaturePipeline

## 1. Univariate correlation screening

In [ ]:
feat_cols = None
for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    pipe = FeaturePipeline(l1=l1, resources_dir=os.path.join(REPO_DIR, 'resources'))
    pipe.fit(train_df)
    X = pipe.transform(train_df)
    y = X['GLMM_score']
    X = X.drop(columns=['GLMM_score'], errors='ignore')
    if feat_cols is None:
        feat_cols = list(X.columns)
    corr_pearson = {c: abs(pearsonr(X[c].fillna(X[c].median()), y)[0]) for c in X.columns}
    corr_spearman = {c: abs(spearmanr(X[c].fillna(X[c].median()), y)[0]) for c in X.columns}
    drop_low = [c for c in X.columns if corr_pearson.get(c, 0) < 0.05 and corr_spearman.get(c, 0) < 0.05]
    anchor = [c for c in X.columns if corr_pearson.get(c, 0) > 0.25 or corr_spearman.get(c, 0) > 0.25]
    print(f'{l1}: drop |r|<0.05: {drop_low[:5]}...; anchor |r|>0.25: {anchor}')

## 2. Multicollinearity check

In [ ]:
train_df = pd.read_csv(TRAIN['es'])
pipe = FeaturePipeline(l1='es', resources_dir=os.path.join(REPO_DIR, 'resources'))
pipe.fit(train_df)
X = pipe.transform(train_df).drop(columns=['GLMM_score'], errors='ignore')
corr = X.corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Feature-feature correlation (train es)')
plt.tight_layout()
plt.savefig(os.path.join(REPO_DIR, 'results/figures/feature_correlation_heatmap.pdf'), bbox_inches='tight')
plt.show()
high = [(i,j,corr.loc[i,j]) for i in corr.index for j in corr.columns if i<j and abs(corr.loc[i,j]) > 0.85]
print('Pairs with |r|>0.85:', high)

## 3. XGBoost permutation importance (training data per L1)

In [ ]:
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance

shared, es_de_specific, cn_specific = [], [], []
for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    pipe = FeaturePipeline(l1=l1, resources_dir=os.path.join(REPO_DIR, 'resources'))
    pipe.fit(train_df)
    X = pipe.transform(train_df)
    y = X['GLMM_score']
    X = X.drop(columns=['GLMM_score'], errors='ignore')
    model = XGBRegressor(n_estimators=200, max_depth=6, random_state=42)
    model.fit(X, y)
    perm = permutation_importance(model, X, y, n_repeats=5, random_state=42)
    imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
    print(f'Top features {l1}:', list(imp.head(10).index))
    if l1 == 'es':
        shared = list(imp.head(15).index)
        es_de_specific = ['edit_distance','is_cognate','char_ngram_overlap','length_ratio']
    if l1 == 'cn':
        cn_specific = ['cn_stroke_complexity'] if 'cn_stroke_complexity' in X.columns else []

## 4. SHAP on XGBoost (verify directions)

In [ ]:
import shap
train_df = pd.read_csv(TRAIN['es'])
pipe = FeaturePipeline(l1='es', resources_dir=os.path.join(REPO_DIR, 'resources'))
pipe.fit(train_df)
X = pipe.transform(train_df).drop(columns=['GLMM_score'], errors='ignore')
y = train_df['GLMM_score']
model = XGBRegressor(n_estimators=200, max_depth=6, random_state=42)
model.fit(X, y)
explainer = shap.TreeExplainer(model, X)
shap_vals = explainer.shap_values(X)
shap.summary_plot(shap_vals, X, max_display=15)

## 5. Freeze features -> frozen_features.json

In [ ]:
shared = ['reveal_ratio','word_length','syllable_count','edit_distance','is_cognate','char_ngram_overlap','context_length','context_ttr','target_position_ratio']
es_de = ['edit_distance','is_cognate','char_ngram_overlap','length_ratio']
cn_spec = ['cn_stroke_complexity']
import json
with open(os.path.join(REPO_DIR, 'frozen_features.json'), 'w') as f:
    json.dump({'shared': shared, 'es_de_specific': es_de, 'cn_specific': cn_spec}, f, indent=2)
print('Wrote frozen_features.json. Adjust shared/es_de/cn_specific as needed.')

In [ ]:
!cd {REPO_DIR} && python scripts/freeze_features.py --shared "reveal_ratio,word_length,syllable_count,context_length,context_ttr,target_position_ratio" --es_de "edit_distance,is_cognate,char_ngram_overlap,length_ratio" --cn "cn_stroke_complexity" -o frozen_features.json

## 6. Sanity: XGBoost with frozen features on dev

In [ ]:
from models.xgboost_baseline import XGBoostBaseline
from scripts.evaluate import CLOSED_BASELINES

for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    dev_df = pd.read_csv(DEV[l1])
    pipe = FeaturePipeline(l1=l1, frozen_features_path=os.path.join(REPO_DIR, 'frozen_features.json'), resources_dir=os.path.join(REPO_DIR, 'resources'))
    baseline = XGBoostBaseline(l1=l1, feature_pipeline=pipe)
    baseline.train(train_df)
    res = baseline.evaluate(dev_df)
    delta = res['rmse'] - CLOSED_BASELINES[l1] if res['rmse'] else None
    print(f'{l1} dev RMSE: {res["rmse"]:.4f}, baseline: {CLOSED_BASELINES[l1]}, beat: {delta < 0 if delta is not None else "N/A"}')